# One portrait, two energies, one Cahn–Hilliard equation

This notebook is for first-year undergraduate students. We treat the red, green and blue channels of a portrait as three scalar fields. Starting from the same disturbed image, we ask how changing the **energy** changes the result.

**image → three fields → energy → chemical potential → Cahn–Hilliard evolution → checks**

All quantities are dimensionless. This is a teaching analogy, not a realistic colour model or a complete image-segmentation method.

## 1. Read the portrait as three fields

Each pixel contains a red, green and blue number. We resize the image for a quick calculation, then map every channel from $0\leq I\leq1$ to a signed field $-1\leq u\leq1$. NumPy handles arrays and Fourier transforms, Matplotlib draws figures, and scikit-image reads the image.

In [ ]:
# In Google Colab, uncomment this once:
# %pip install -q scikit-image
import numpy as np
import matplotlib.pyplot as plt
from skimage import color, io, transform

portrait_url = 'https://saswataiith.github.io/assets/images/saswata-landscape.png'
portrait = io.imread(portrait_url)
if portrait.shape[-1] == 4:
    portrait = color.rgba2rgb(portrait)
portrait = transform.resize(portrait, (128, 156), anti_aliasing=True)
target = 2.0 * portrait - 1.0

plt.imshow(portrait)
plt.axis('off')
plt.title('The target portrait')
plt.show()

## 2. Disturb the image without changing its mean

We add random noise to break up the portrait. For each colour channel we subtract the mean noise. The disturbance therefore does not change the total amount of that channel. This matters because Cahn–Hilliard dynamics conserves spatial averages. The seed only makes the exercise repeatable.

In [ ]:
random_generator = np.random.default_rng(20260922)
noise = random_generator.normal(0.0, 0.32, target.shape)
noise -= noise.mean(axis=(0, 1), keepdims=True)
initial = target + noise

def show_rgb(field):
    return np.clip((field + 1.0) / 2.0, 0.0, 1.0)

plt.imshow(show_rgb(initial))
plt.axis('off')
plt.title('The common starting field')
plt.show()

## 3. Choose the energy

For the **portrait energy**, $r_j$ is the target and $u_j$ is the evolving field:

$$F_{portrait}=\frac12\int\sum_j(u_j-r_j)^2\,dA.$$

It is zero only when the portrait returns. For the **domain energy**, every channel prefers values near $-1$ or $+1$, while the gradient term penalizes sharp boundaries:

$$F_{domains}=\int\sum_j\left[\frac{(u_j^2-1)^2}{4}+\frac{\kappa}{2}|\nabla u_j|^2\right]dA.$$

The two calculations start from the same field and follow the same type of dynamics, but their energies specify different destinations.

## 4. Fourier derivatives

A Fourier transform represents a field as waves. The Laplacian $\nabla^2u$ becomes multiplication of each Fourier coefficient by $-k^2$. This gives short, accurate periodic derivatives. The $k=0$ mode is the spatial average. Cahn–Hilliard multiplies its rate by $k^2=0$, so that average cannot change.

In [ ]:
rows, columns, channels = target.shape
wave_y = 2.0 * np.pi * np.fft.fftfreq(rows)
wave_x = 2.0 * np.pi * np.fft.fftfreq(columns)
wave_x, wave_y = np.meshgrid(wave_x, wave_y)
wave_number_squared = (wave_x**2 + wave_y**2)[..., None]

def to_fourier(field):
    return np.fft.fft2(field, axes=(0, 1))

def to_real(field_hat):
    return np.fft.ifft2(field_hat, axes=(0, 1)).real

## 5. Evolve both systems

Cahn–Hilliard dynamics is

$$\frac{\partial u_j}{\partial t}=M\nabla^2\mu_j,\qquad \mu_j=\frac{\delta F}{\delta u_j}.$$

The chemical potential $\mu_j$ measures how the energy changes when the field changes locally. The outer Laplacian moves the field while conserving its mean. The portrait problem is linear, so we advance it exactly. The domain problem is nonlinear, so we use a stable semi-implicit step. Every operation is shown below.

In [ ]:
mobility = 1.0
gradient_coefficient = 0.4
time_step = 0.1
stabilization = 2.0
number_of_steps = 800
save_every = 100

noise_hat = to_fourier(noise)
domain_field = initial.copy()
portrait_frames, domain_frames, times = [], [], []

for iterator in range(number_of_steps + 1):
    time = iterator * time_step
    portrait_field = target + to_real(
        noise_hat * np.exp(-mobility * wave_number_squared * time)
    )

    if iterator % save_every == 0:
        portrait_frames.append(portrait_field.copy())
        domain_frames.append(domain_field.copy())
        times.append(time)

    if iterator < number_of_steps:
        nonlinear_term = domain_field**3 - domain_field
        numerator = to_fourier(domain_field) - (
            time_step * mobility * wave_number_squared
            * to_fourier(nonlinear_term - stabilization * domain_field)
        )
        denominator = 1.0 + time_step * mobility * wave_number_squared * (
            stabilization + gradient_coefficient * wave_number_squared
        )
        domain_field = to_real(numerator / denominator)

## 6. Look at early, middle and late times

The first row returns toward the prescribed portrait. The second forms domains because its double-well energy prefers two field values. Neither result is pasted in at the end.

In [ ]:
chosen_frames = [0, len(times) // 2, len(times) - 1]
figure, axes = plt.subplots(2, 3, figsize=(11, 6))
for column, frame_number in enumerate(chosen_frames):
    axes[0, column].imshow(show_rgb(portrait_frames[frame_number]))
    axes[1, column].imshow(show_rgb(domain_frames[frame_number]))
    axes[0, column].set_title(f'portrait energy, t = {times[frame_number]:.0f}')
    axes[1, column].set_title(f'domain energy, t = {times[frame_number]:.0f}')
    axes[0, column].axis('off')
    axes[1, column].axis('off')
plt.tight_layout()
plt.show()

## 7. Check the physics, not only the pictures

We calculate the **mean energy density**, meaning total dimensionless energy divided by image area and three channels. We also test conservation. Because the two functionals differ, compare whether each energy falls rather than comparing their absolute sizes.

In [ ]:
def portrait_energy(field):
    return 0.5 * np.mean((field - target)**2)

def domain_energy(field):
    local_energy = np.mean((field**2 - 1.0)**2 / 4.0)
    field_hat = to_fourier(field)
    gradient_energy = gradient_coefficient * np.sum(
        wave_number_squared * np.abs(field_hat)**2
    ) / (2.0 * channels * (rows * columns)**2)
    return local_energy + gradient_energy

portrait_energies = np.array([portrait_energy(field) for field in portrait_frames])
domain_energies = np.array([domain_energy(field) for field in domain_frames])
initial_means = initial.mean(axis=(0, 1))
largest_mean_drift = max(
    np.max(np.abs(field.mean(axis=(0, 1)) - initial_means))
    for field in portrait_frames + domain_frames
)

plt.semilogy(times, portrait_energies, 'o-', label='portrait energy')
plt.semilogy(times, domain_energies, 's-', label='domain energy')
plt.xlabel('dimensionless time')
plt.ylabel('mean energy density')
plt.legend()
plt.show()

assert np.all(np.diff(portrait_energies) <= 1e-12)
assert np.all(np.diff(domain_energies) <= 1e-10)
assert largest_mean_drift < 1e-12
print(f'Largest drift of any channel mean: {largest_mean_drift:.2e}')
print('Checks passed: both energies fall and all channel means are conserved.')

## 8. Try your own computational experiments

Change one number at a time and predict the result before running it.

1. Increase `gradient_coefficient`. Do domain boundaries become wider or narrower?
2. Reduce `mobility`. Does it change the preferred final state, or only the speed?
3. Increase the noise amplitude. Are the channel means still conserved?
4. Replace the portrait with another public image URL.
5. Why do short and long wavelength disturbances decay at different rates?

**The energy tells us where the system prefers to go. The evolution equation tells us how it can get there.**